# F1: переоценка сохранённых адаптеров на БОЛЬШОМ тесте
(ROADMAP F1; без обучения — только инференс best_prefix.pt)

- **D1**: все 2542 тест-эпизода ED (было 500) — gen- и verb-accuracy, per-example.
- **D2**: первые 1000 диалогов (было 500) — rougeL per-example, Distinct.
- Протокол идентичен экспериментам (легаси-паддинг 0 в коллаторе,
  strip→pad_token_id, greedy), поэтому числа сопоставимы со старыми.
- CI разностей сужается ~×2 → пограничные эффекты (mask в D1) разрешимы.
- Время: D1 ~1.5 ч + D2 ~3.5 ч. Resume-safe (каждый прогон -> json).

Порядок: ячейки сверху вниз.

In [1]:
import os, gc, json, time, re
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import PrefixTuningConfig, TaskType, get_peft_model
from datasets import Dataset as HFDataset, load_dataset
from huggingface_hub import list_repo_files, hf_hub_download
import numpy as np
import evaluate

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "./eval_bigtest"
os.makedirs(OUT_DIR, exist_ok=True)
D2_TEST_N = 1000          # из 2540 доступных диалогов
REPO, REV = "empathetic_dialogues", "refs/convert/parquet"

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B")
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("tokenizer ok; dev:", DEV)


def collator(features):
    """Легаси-конвенция experiment.ipynb: input_ids паддится 0,
    attention_mask — 0, labels — -100 (левый паддинг)."""
    def t1(x):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.long)
        return x.squeeze(0) if x.dim() > 1 else x
    def pad(seqs, value):
        m = max(s.size(0) for s in seqs)
        return torch.stack([s if s.size(0) == m
                            else F.pad(s, (m - s.size(0), 0), value=value)
                            for s in seqs])
    out = {"input_ids": pad([t1(f["input_ids"]) for f in features], 0),
           "attention_mask": pad([t1(f["attention_mask"]) for f in features], 0)}
    m = out["input_ids"].size(1)
    out["labels"] = torch.stack([
        t1(f["labels"]) if t1(f["labels"]).size(0) == m
        else F.pad(t1(f["labels"]), (m - t1(f["labels"]).size(0), 0), value=-100)
        for f in features])
    return out


def strip_label_tokens(batch):
    """Как в experiment.ipynb: отрезать метку/ответ, перепаддить слева
    pad_token_id (eval-конвенция)."""
    labels = batch["labels"]
    n_lab = (labels != -100).sum(dim=1)
    L = batch["input_ids"].size(1)
    srcs = [batch["input_ids"][i, :L - int(n_lab[i])]
            for i in range(batch["input_ids"].size(0))]
    ams = [batch["attention_mask"][i, :L - int(n_lab[i])]
           for i in range(batch["input_ids"].size(0))]
    max_len = max(s.size(0) for s in srcs)
    input_ids = torch.stack([
        s if s.size(0) == max_len
        else F.pad(s, (max_len - s.size(0), 0), value=tok.pad_token_id)
        for s in srcs])
    attention_mask = torch.stack([
        m if m.size(0) == max_len
        else F.pad(m, (max_len - m.size(0), 0), value=0)
        for m in ams])
    return input_ids, attention_mask


def distinct_n(texts):
    unis, bis = set(), set()
    nt, np_ = 0, 0
    for t in texts:
        w = t.split()
        unis.update(w); bis.update(zip(w, w[1:]))
        nt += len(w); np_ += max(len(w) - 1, 0)
    return len(unis) / max(nt, 1), len(bis) / max(np_, 1)


def load_eval_model(model_name):
    set_seed(42)
    base = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=torch.bfloat16, trust_remote_code=True)
    model = get_peft_model(
        base, PrefixTuningConfig(task_type=TaskType.CAUSAL_LM,
                                 num_virtual_tokens=20,
                                 prefix_projection=True, inference_mode=False))
    model.enable_input_require_grads()
    model.to(DEV).eval()
    return model


def swap_adapter(model, path):
    st = torch.load(path, map_location="cpu")
    n = 0
    with torch.no_grad():
        for name, p in model.named_parameters():
            if name in st:
                p.data.copy_(st[name].to(p.device)); n += 1
    return n

print("хелперы готовы")

<VENV>/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tokenizer ok; dev: cuda
хелперы готовы


In [2]:
# ---------- D1: данные — ВСЕ тест-эпизоды (2542) ----------
def load_ed_episodes(split):
    files = list_repo_files(REPO, repo_type="dataset", revision=REV)
    names = sorted(f for f in files
                   if f.startswith(f"default/{split}/") and f.endswith(".parquet"))
    paths = [hf_hub_download(REPO, n, repo_type="dataset", revision=REV) for n in names]
    ds = load_dataset("parquet", data_files=paths, split="train")
    episodes, seen = [], set()
    for r in ds:
        cid = r["conv_id"]
        if cid in seen:
            continue
        seen.add(cid)
        t, l = r["prompt"].strip(), r["context"].strip()
        if t and l:
            episodes.append({"text": t, "label": l})
    return episodes

ed_test_full = load_ed_episodes("test")
emotion_labels = sorted({e["label"] for e in ed_test_full})
assert len(emotion_labels) == 32

def _norm(s):
    return re.sub(r"[^a-z ]", "", s.lower()).strip()
EMOTION_BY_NORM = {_norm(l): l for l in emotion_labels}

def match_emotion(text):
    g = _norm(text)
    if not g:
        return None
    first = g.split()[0]
    if first in EMOTION_BY_NORM:
        return EMOTION_BY_NORM[first]
    if g in EMOTION_BY_NORM:
        return EMOTION_BY_NORM[g]
    for nl, l in EMOTION_BY_NORM.items():
        if g.startswith(nl):
            return l
    for nl, l in EMOTION_BY_NORM.items():
        if nl in g:
            return l
    return None

LABEL_FIRST_TOKEN = {l: tok(" " + l, add_special_tokens=False,
                            truncation=True, max_length=8)["input_ids"][0]
                     for l in emotion_labels}
assert len(set(LABEL_FIRST_TOKEN.values())) == 32
references_d1 = [e["label"] for e in ed_test_full]

def tokenize_d1_eval(examples):
    ii, am, ll = [], [], []
    for s, t in zip(examples["text"], examples["label"]):
        src = tok(f"Situation: {s}\nEmotion:", add_special_tokens=False,
                  truncation=True, max_length=96)["input_ids"] or [tok.pad_token_id]
        tgt = tok(" " + t, add_special_tokens=False, truncation=True,
                  max_length=8)["input_ids"]
        ii.append(src + tgt); am.append([1] * (len(src) + len(tgt)))
        ll.append([-100] * len(src) + tgt)
    return {"input_ids": ii, "attention_mask": am, "labels": ll}

test_d1 = HFDataset.from_list(ed_test_full).map(
    tokenize_d1_eval, batched=True,
    remove_columns=["text", "label"]).with_format("torch")
print(f"D1 тест: {len(test_d1)} эпизодов (было 500), 32 класса")

Map: 100%|██████████| 2542/2542 [00:00<00:00, 8637.08 examples/s]

D1 тест: 2542 эпизодов (было 500), 32 класса


In [3]:
# ---------- D1: переоценка прогонов ----------
D1_RUNS = {"MLE-only": "gen_crl_mle_baseline",
           "contrast-mask": "gen_dialogue_ed_contrast_mask",
           "contrast-wronglabel": "gen_dialogue_ed_contrast_wl",
           "RL-only": "gen_dialogue_ed_rl_only",
           "Full-CRL": "gen_dialogue_ed_full_crl"}
D1_SEEDS = list(range(42, 52))   # 42-48 готовы; 49-51 — продление RL-пары (§6.5), отсутствующие пропускаются

@torch.no_grad()
def eval_d1(model):
    model.eval()
    order = sorted(LABEL_FIRST_TOKEN)
    tok_ids = torch.tensor([LABEL_FIRST_TOKEN[l] for l in order], device=DEV)
    loader = DataLoader(test_d1, batch_size=32, collate_fn=collator)
    gen_ok, verb_ok = [], []
    idx = 0                     # глобальный индекс примера
    for batch in loader:
        ii, am = strip_label_tokens(batch)
        ii, am = ii.to(DEV), am.to(DEV)
        logits = model(input_ids=ii, attention_mask=am).logits[:, -1, :]
        for pi in logits[:, tok_ids].argmax(dim=-1).tolist():
            verb_ok.append(int(order[pi] == references_d1[idx]))
            idx += 1
        gen = model.generate(input_ids=ii, attention_mask=am, max_new_tokens=6,
                             do_sample=False, num_beams=1,
                             pad_token_id=tok.pad_token_id,
                             eos_token_id=tok.eos_token_id)
        for j in range(ii.size(0)):
            pred = tok.decode(gen[j, ii.shape[1]:], skip_special_tokens=True).strip()
            gen_ok.append(int(match_emotion(pred) == references_d1[len(gen_ok)]))
    return gen_ok, verb_ok

model = load_eval_model("Qwen/Qwen2.5-3B")
d1_big = {}
for name, base_dir in D1_RUNS.items():
    d1_big[name] = {}
    for seed in D1_SEEDS:
        run_dir = f"{base_dir}_d16_seed{seed}"
        adapter = os.path.join(run_dir, "best_prefix.pt")
        cache = f"{OUT_DIR}/d1_{name}_s{seed}.json"
        if os.path.exists(cache):
            with open(cache) as f:
                d1_big[name][seed] = json.load(f)
            continue
        if not os.path.exists(adapter):
            print(f"  [skip] {name} s{seed}: нет адаптера"); continue
        n_sw = swap_adapter(model, adapter)
        t0 = time.time()
        gen_ok, verb_ok = eval_d1(model)
        rec = {"accuracy": float(np.mean(gen_ok)),
               "verb_accuracy": float(np.mean(verb_ok)),
               "correct_vec": gen_ok, "verb_correct_vec": verb_ok}
        with open(cache, "w") as f:
            json.dump(rec, f)
        d1_big[name][seed] = rec
        print(f"  {name} s{seed}: acc={rec['accuracy']:.4f} "
              f"verb={rec['verb_accuracy']:.4f} (n={len(gen_ok)}, "
              f"{time.time()-t0:.0f}s, {n_sw} тензоров)")
del model; gc.collect(); torch.cuda.empty_cache()
print("D1 переоценка завершена")

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 8896.01it/s]


  MLE-only s49: acc=0.6145 verb=0.6129 (n=2542, 74s, 5 тензоров)
  MLE-only s50: acc=0.6192 verb=0.6168 (n=2542, 74s, 5 тензоров)
  MLE-only s51: acc=0.6054 verb=0.5983 (n=2542, 74s, 5 тензоров)
  [skip] contrast-mask s49: нет адаптера
  [skip] contrast-mask s50: нет адаптера
  [skip] contrast-mask s51: нет адаптера
  [skip] contrast-wronglabel s49: нет адаптера
  [skip] contrast-wronglabel s50: нет адаптера
  [skip] contrast-wronglabel s51: нет адаптера
  RL-only s49: acc=0.6086 verb=0.6125 (n=2542, 74s, 5 тензоров)
  RL-only s50: acc=0.6192 verb=0.6129 (n=2542, 74s, 5 тензоров)
  RL-only s51: acc=0.6267 verb=0.6227 (n=2542, 74s, 5 тензоров)
  [skip] Full-CRL s49: нет адаптера
  [skip] Full-CRL s50: нет адаптера
  [skip] Full-CRL s51: нет адаптера
D1 переоценка завершена


In [4]:
# ---------- D1: статистика на большом тесте ----------
from math import sqrt
from scipy.stats import t as _t_dist
# критерий из распределения Стьюдента (критика-3 §28: ручной словарь
# с fallback 2.447 неверен при n>7 — например n=10 требует 2.262)
def tcrit_of(n):
    return float(_t_dist.ppf(0.975, df=n - 1))

def table_and_pairs(big, runs, seeds, vec_key, label):
    names = list(runs)
    print("\n" + "=" * 84)
    print(f"{label} (большой тест)")
    print("=" * 84)
    for n in names:
        vals = [big[n][s][{"correct_vec": "accuracy",
                           "per_example_rougeL": "rougeL"}[vec_key]]
                for s in seeds if s in big[n]]
        print(f"  {n:<22} {np.mean(vals):.4f} ± {np.std(vals):.4f}  "
              f"{[round(v, 4) for v in vals]}")
    pairs = [("contrast-mask", "MLE-only"), ("RL-only", "MLE-only"),
             ("Full-CRL", "MLE-only"), ("Full-CRL", "RL-only"),
             ("contrast-wronglabel", "contrast-mask")]
    print("\nПары (t по сидам + бутстреп по примерам):")
    for a, b in pairs:
        if a not in big or b not in big:
            continue
        common = [s for s in seeds if s in big[a] and s in big[b]]
        d = [big[a][s][{"correct_vec": "accuracy",
                        "per_example_rougeL": "rougeL"}[vec_key]]
             - big[b][s][{"correct_vec": "accuracy",
                          "per_example_rougeL": "rougeL"}[vec_key]]
             for s in common]
        se = np.std(d, ddof=1) / sqrt(len(d)) if len(d) > 1 else float("nan")
        t = np.mean(d) / se if se and se > 0 else float("nan")
        diffs = []
        for s in common:
            diffs += [x - y for x, y in zip(big[a][s][vec_key], big[b][s][vec_key])]
        arr = np.array(diffs, float)
        rng = np.random.default_rng(42)
        boots = np.array([arr[rng.integers(0, len(arr), len(arr))].mean()
                          for _ in range(10000)])
        lo, hi = np.percentile(boots, [2.5, 97.5])
        p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
        sig = abs(t) > tcrit_of(len(d)) and p < 0.05
        print(f"  {a:<20}−{b:<14}: {np.mean(d):+.4f}±{np.std(d, ddof=1):.4f} "
              f"t={t:+.2f} p_boot={p:.4f} CI[{lo:+.4f},{hi:+.4f}] "
              f"{'ЗНАЧИМО' if sig else 'н.з.'} n_пар={len(arr)}")

table_and_pairs(d1_big, D1_RUNS, D1_SEEDS, "correct_vec",
                f"D1: классификация эмоции (ED, тест 2542, n_сидов={len(seeds)})")
with open(f"{OUT_DIR}/summary_d1_big.json", "w") as f:
    json.dump({n: {str(s): {"accuracy": r["accuracy"],
                            "verb_accuracy": r["verb_accuracy"]}
                   for s, r in per.items()} for n, per in d1_big.items()}, f, indent=2)
print(f"\nСохранено: {OUT_DIR}/summary_d1_big.json")


D1: классификация эмоции (ED, тест 2542, n_сидов=7) (большой тест)
  MLE-only               0.6204 ± 0.0072  [0.6192, 0.6188, 0.6227, 0.6247, 0.6353, 0.6212, 0.6227, 0.6145, 0.6192, 0.6054]
  contrast-mask          0.6233 ± 0.0056  [0.6157, 0.6204, 0.6334, 0.6176, 0.6279, 0.6247, 0.6235]
  contrast-wronglabel    0.5695 ± 0.1175  [0.6161, 0.6157, 0.6279, 0.6286, 0.62, 0.5956, 0.2828]
  RL-only                0.6247 ± 0.0074  [0.6318, 0.6196, 0.6259, 0.6263, 0.6377, 0.6275, 0.6239, 0.6086, 0.6192, 0.6267]
  Full-CRL               0.6232 ± 0.0095  [0.6223, 0.6176, 0.6318, 0.6042, 0.6334, 0.6314, 0.622]

Пары (t по сидам + бутстреп по примерам):
  contrast-mask       −MLE-only      : -0.0002±0.0064 t=-0.09 p_boot=0.9270 CI[-0.0044,+0.0039] н.з. n_пар=17794
  RL-only             −MLE-only      : +0.0043±0.0076 t=+1.81 p_boot=0.0230 CI[+0.0006,+0.0080] н.з. n_пар=25420
  Full-CRL            −MLE-only      : -0.0003±0.0102 t=-0.07 p_boot=0.9224 CI[-0.0046,+0.0040] н.з. n_пар=17794
  Full-CRL

In [5]:
# ---------- D2: данные — первые D2_TEST_N диалогов ----------
def load_ed_conversations(split):
    files = list_repo_files(REPO, repo_type="dataset", revision=REV)
    names = sorted(f for f in files
                   if f.startswith(f"default/{split}/") and f.endswith(".parquet"))
    paths = [hf_hub_download(REPO, n, repo_type="dataset", revision=REV) for n in names]
    ds = load_dataset("parquet", data_files=paths, split="train")
    by_conv = {}
    for r in ds:
        by_conv.setdefault(r["conv_id"], []).append(r)
    convs = []
    for cid in sorted(by_conv):
        rows = sorted(by_conv[cid], key=lambda r: r["utterance_idx"])
        spk0 = rows[0]["speaker_idx"]
        reply = next((r for r in rows if r["speaker_idx"] != spk0), None)
        if reply is None:
            continue
        utt = reply["utterance"].replace("_comma_", ",").strip()
        sit, emo = rows[0]["prompt"].strip(), rows[0]["context"].strip()
        if utt and sit and emo:
            convs.append({"emotion": emo, "situation": sit, "response": utt})
    return convs

d2_rows = load_ed_conversations("test")[:D2_TEST_N]
references_d2 = [r["response"] for r in d2_rows]

def tokenize_d2_eval(examples):
    ii, am, ll = [], [], []
    for emo, sit, resp in zip(examples["emotion"], examples["situation"],
                              examples["response"]):
        src = tok(f"Emotion: {emo}\nSituation: {sit}\nResponse:",
                  add_special_tokens=False, truncation=True,
                  max_length=128)["input_ids"] or [tok.pad_token_id]
        tgt = tok(" " + resp, add_special_tokens=False, truncation=True,
                  max_length=47)["input_ids"] + [tok.eos_token_id]
        ii.append(src + tgt); am.append([1] * (len(src) + len(tgt)))
        ll.append([-100] * len(src) + tgt)
    return {"input_ids": ii, "attention_mask": am, "labels": ll}

test_d2 = HFDataset.from_list(d2_rows).map(
    tokenize_d2_eval, batched=True,
    remove_columns=["emotion", "situation", "response"]).with_format("torch")
print(f"D2 тест: {len(test_d2)} диалогов (было 500)")

Map: 100%|██████████| 1000/1000 [00:00<00:00, 7134.44 examples/s]

D2 тест: 1000 диалогов (было 500)


In [6]:
# ---------- D2: переоценка прогонов ----------
D2_RUNS = [("MLE-only", "gen_d2_mle", "d21", D1_SEEDS),
           ("contrast-mask", "gen_d2_mask", "d21", D1_SEEDS),
           ("contrast-offtopic", "gen_d2_offtopic", "d21", D1_SEEDS),
           ("contrast-incoherent", "gen_d2_incoherent", "d21", D1_SEEDS),
           ("RL-only", "gen_d2_rl_only", "d22", D1_SEEDS),
           ("offtopic4", "gen_d2_offtopic4", "d23", [42, 43, 44])]
rouge_metric = evaluate.load("rouge")

@torch.no_grad()
def eval_d2(model):
    model.eval()
    loader = DataLoader(test_d2, batch_size=8, collate_fn=collator)
    preds = []
    for batch in loader:
        ii, am = strip_label_tokens(batch)
        ii, am = ii.to(DEV), am.to(DEV)
        gen = model.generate(input_ids=ii, attention_mask=am, max_new_tokens=48,
                             do_sample=False, num_beams=1,
                             pad_token_id=tok.pad_token_id,
                             eos_token_id=tok.eos_token_id)
        for j in range(ii.size(0)):
            preds.append(tok.decode(gen[j, ii.shape[1]:],
                                    skip_special_tokens=True).strip() or " ")
    per_ex = [float(x) for x in rouge_metric.compute(
        predictions=preds, references=references_d2,
        rouge_types=["rougeL"], use_aggregator=False)["rougeL"]]
    return per_ex, preds

model = load_eval_model("Qwen/Qwen2.5-3B")
d2_big = {}
for name, base_dir, wave, seeds in D2_RUNS:
    d2_big[name] = {}
    for seed in seeds:
        run_dir = f"{base_dir}_{wave}_seed{seed}"
        adapter = os.path.join(run_dir, "best_prefix.pt")
        cache = f"{OUT_DIR}/d2_{name}_s{seed}.json"
        if os.path.exists(cache):
            with open(cache) as f:
                d2_big[name][seed] = json.load(f)
            continue
        if not os.path.exists(adapter):
            print(f"  [skip] {name} s{seed}"); continue
        swap_adapter(model, adapter)
        t0 = time.time()
        per_ex, preds = eval_d2(model)
        d1v, d2v = distinct_n(preds)
        rec = {"rougeL": float(np.mean(per_ex)), "per_example_rougeL": per_ex,
               "distinct1": d1v, "distinct2": d2v}
        with open(cache, "w") as f:
            json.dump(rec, f)
        d2_big[name][seed] = rec
        print(f"  {name} s{seed}: rougeL={rec['rougeL']:.4f} d1={d1v:.4f} "
              f"({time.time()-t0:.0f}s)")
del model; gc.collect(); torch.cuda.empty_cache()
print("D2 переоценка завершена")

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 10740.41it/s]


  MLE-only s42: rougeL=0.1696 d1=0.0909 (93s)
  MLE-only s43: rougeL=0.1630 d1=0.0976 (108s)
  MLE-only s44: rougeL=0.1661 d1=0.0992 (105s)
  MLE-only s45: rougeL=0.1634 d1=0.0978 (114s)
  MLE-only s46: rougeL=0.1724 d1=0.0965 (87s)
  MLE-only s47: rougeL=0.1681 d1=0.0948 (89s)
  MLE-only s48: rougeL=0.1660 d1=0.0890 (110s)
  contrast-mask s42: rougeL=0.1686 d1=0.0906 (97s)
  contrast-mask s43: rougeL=0.1637 d1=0.0986 (110s)
  contrast-mask s44: rougeL=0.1742 d1=0.1035 (80s)
  contrast-mask s45: rougeL=0.1671 d1=0.1001 (91s)
  contrast-mask s46: rougeL=0.1716 d1=0.0976 (84s)
  contrast-mask s47: rougeL=0.1669 d1=0.0985 (100s)
  contrast-mask s48: rougeL=0.1262 d1=0.0261 (111s)
  contrast-offtopic s42: rougeL=0.1699 d1=0.0903 (90s)
  contrast-offtopic s43: rougeL=0.1634 d1=0.0988 (104s)
  contrast-offtopic s44: rougeL=0.1577 d1=0.0914 (111s)
  contrast-offtopic s45: rougeL=0.1696 d1=0.1007 (94s)
  contrast-offtopic s46: rougeL=0.1674 d1=0.0952 (87s)
  contrast-offtopic s47: rougeL=0.170

In [7]:
# ---------- D2: статистика на большом тесте ----------
table_and_pairs(d2_big, {n: b for n, b, w, s in D2_RUNS},
               D1_SEEDS, "per_example_rougeL",
               f"D2: генерация ответов (ED, тест {D2_TEST_N}, n_сидов=7)")
print("\nDistinct-1 (mean):", {n: round(float(np.mean(
    [d2_big[n][s]["distinct1"] for s in d2_big[n]])), 4) for n in d2_big})
with open(f"{OUT_DIR}/summary_d2_big.json", "w") as f:
    json.dump({n: {str(s): {"rougeL": r["rougeL"], "distinct1": r["distinct1"]}
                   for s, r in per.items()} for n, per in d2_big.items()}, f, indent=2)
print(f"Сохранено: {OUT_DIR}/summary_d2_big.json")


D2: генерация ответов (ED, тест 1000, n_сидов=7) (большой тест)
  MLE-only               0.1670 ± 0.0031  [0.1696, 0.163, 0.1661, 0.1634, 0.1724, 0.1681, 0.166]
  contrast-mask          0.1626 ± 0.0152  [0.1686, 0.1637, 0.1742, 0.1671, 0.1716, 0.1669, 0.1262]
  contrast-offtopic      0.1659 ± 0.0043  [0.1699, 0.1634, 0.1577, 0.1696, 0.1674, 0.1702, 0.1634]
  contrast-incoherent    0.1601 ± 0.0161  [0.1215, 0.1695, 0.1685, 0.1697, 0.1608, 0.163, 0.1678]
  RL-only                0.1673 ± 0.0031  [0.1729, 0.1657, 0.1633, 0.1656, 0.1688, 0.1699, 0.1647]
  offtopic4              0.1666 ± 0.0042  [0.1724, 0.1626, 0.1647]

Пары (t по сидам + бутстреп по примерам):
  contrast-mask       −MLE-only      : -0.0043±0.0160 t=-0.72 p_boot=0.0004 CI[-0.0066,-0.0022] н.з. n_пар=7000
  RL-only             −MLE-only      : +0.0003±0.0028 t=+0.29 p_boot=0.7464 CI[-0.0016,+0.0022] н.з. n_пар=7000

Distinct-1 (mean): {'MLE-only': 0.0951, 'contrast-mask': 0.0878, 'contrast-offtopic': 0.0961, 'contrast-inco